In [ ]:
!pip install torch torch-geometric scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.9 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cora.cites to cora.cites


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cora.content to cora.content


In [ ]:
import torch
import numpy as np

# cora.content: node_id | word features | label
content = np.genfromtxt("cora.content", dtype=str)

node_ids = content[:, 0]
features = content[:, 1:-1].astype(float)

# map node id → index
id_map = {node_id: i for i, node_id in enumerate(node_ids)}

x = torch.tensor(features, dtype=torch.float)

In [ ]:
# cora.cites: cited_paper  citing_paper
edges = np.genfromtxt("cora.cites", dtype=str)

edge_index = []
for src, dst in edges:
    edge_index.append([id_map[src], id_map[dst]])
    edge_index.append([id_map[dst], id_map[src]])  # make undirected

edge_index = torch.tensor(edge_index, dtype=torch.long).t()

In [ ]:
from sklearn.model_selection import train_test_split

num_edges = edge_index.size(1) // 2
all_edges = edge_index[:, :num_edges].t()

train_edges, test_edges = train_test_split(
    all_edges, test_size=0.2, random_state=42)

val_edges, test_edges = train_test_split(
    test_edges, test_size=0.5, random_state=42)

In [ ]:
import random

def negative_sampling(num_samples, num_nodes, edge_set):
    neg_edges = []
    while len(neg_edges) < num_samples:
        u = random.randint(0, num_nodes - 1)
        v = random.randint(0, num_nodes - 1)
        if u != v and (u, v) not in edge_set:
            neg_edges.append([u, v])
    return torch.tensor(neg_edges)

In [ ]:
num_nodes = x.size(0)
edge_set = set(map(tuple, all_edges.tolist()))

train_neg = negative_sampling(len(train_edges), num_nodes, edge_set)
val_neg   = negative_sampling(len(val_edges), num_nodes, edge_set)
test_neg  = negative_sampling(len(test_edges), num_nodes, edge_set)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv

class GIN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()

        nn1 = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        nn2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.conv1 = GINConv(nn1)
        self.conv2 = GINConv(nn2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [ ]:
def decode(z, edges):
    return (z[edges[:, 0]] * z[edges[:, 1]]).sum(dim=1)

In [ ]:
from sklearn.metrics import average_precision_score

model = GIN(x.size(1), 128).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

x = x.cuda()
edge_index = edge_index.cuda()

def train():
    model.train()
    optimizer.zero_grad()

    z = model(x, edge_index)

    pos_out = decode(z, train_edges.cuda())
    neg_out = decode(z, train_neg.cuda())

    y = torch.cat([
        torch.ones(pos_out.size(0)),
        torch.zeros(neg_out.size(0))
    ]).cuda()

    out = torch.cat([pos_out, neg_out])
    loss = F.binary_cross_entropy_with_logits(out, y)

    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
@torch.no_grad()
def evaluate(pos_edges, neg_edges):
    model.eval()
    z = model(x, edge_index)

    pos_out = decode(z, pos_edges.cuda())
    neg_out = decode(z, neg_edges.cuda())

    scores = torch.cat([pos_out, neg_out]).sigmoid().cpu()
    labels = torch.cat([
        torch.ones(pos_out.size(0)),
        torch.zeros(neg_out.size(0))
    ])

    ap = average_precision_score(labels, scores)
    return ap

In [ ]:
for epoch in range(1, 201):
    loss = train()

    if epoch % 10 == 0:
        val_ap = evaluate(val_edges, val_neg)
        print(f"Epoch {epoch:03d}, Loss {loss:.4f}, Val AP {val_ap:.4f}")

Epoch 010, Loss 0.5046, Val AP 0.9434
Epoch 020, Loss 0.4367, Val AP 0.9676
Epoch 030, Loss 0.3922, Val AP 0.9761
Epoch 040, Loss 0.3475, Val AP 0.9760
Epoch 050, Loss 0.3096, Val AP 0.9761
Epoch 060, Loss 0.2858, Val AP 0.9688
Epoch 070, Loss 0.2415, Val AP 0.9665
Epoch 080, Loss 0.2006, Val AP 0.9537
Epoch 090, Loss 0.1720, Val AP 0.9469
Epoch 100, Loss 0.1398, Val AP 0.9339
Epoch 110, Loss 0.1169, Val AP 0.9345
Epoch 120, Loss 0.0953, Val AP 0.9281
Epoch 130, Loss 0.2141, Val AP 0.9325
Epoch 140, Loss 0.1316, Val AP 0.9267
Epoch 150, Loss 0.0928, Val AP 0.9288
Epoch 160, Loss 0.0672, Val AP 0.9103
Epoch 170, Loss 0.0499, Val AP 0.8934
Epoch 180, Loss 0.0369, Val AP 0.8778
Epoch 190, Loss 0.0269, Val AP 0.8611
Epoch 200, Loss 0.0189, Val AP 0.8455


In [ ]:
test_ap = evaluate(test_edges, test_neg)
print("Test AP:", test_ap)

Test AP: 0.8470416894171857
